In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone llama.cpp
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q gguf

In [ ]:
# 3. Download base model config files only (no weights needed)
from huggingface_hub import hf_hub_download
import os

HF_TOKEN = ''  # paste your HF token
BASE_DIR = '/content/base-config'
os.makedirs(BASE_DIR, exist_ok=True)

for f in ['config.json', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json']:
    hf_hub_download('meta-llama/Meta-Llama-3.1-8B-Instruct', f, token=HF_TOKEN, local_dir=BASE_DIR)
    print('Downloaded', f)

In [ ]:
# 4. Convert LoRA adapter to GGUF
import os

ADAPTER = '/content/drive/MyDrive/Lumen/lumen-checkpoints/checkpoint-2643'
BASE_DIR = '/content/base-config'
OUT     = '/content/lumen-adapter.gguf'

!python /content/llama.cpp/convert_lora_to_gguf.py {ADAPTER} --base {BASE_DIR} --outfile {OUT}

if os.path.exists(OUT) and os.path.getsize(OUT) > 1e6:
    print('Adapter GGUF done. Size:', round(os.path.getsize(OUT)/1e6, 1), 'MB')
else:
    print('ERROR: adapter GGUF not created.')

In [ ]:
# 5. Upload adapter to HuggingFace
from huggingface_hub import HfApi

HF_TOKEN = ''  # paste your HF token
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj='/content/lumen-adapter.gguf',
    path_in_repo='lumen-adapter.gguf',
    repo_id='RavikxxBGamin/Lumen',
    repo_type='model',
)
print('Uploaded to RavikxxBGamin/Lumen!')